# 05 Benchmark: Full-Dataset HNSW

This notebook reproduces the established full-dataset benchmark using the existing handwritten HNSW implementation. Exact NumPy search is the ground truth. The HNSW graph is built once and reused for every `ef_search` value.

Recall@10 is not accuracy: it is the fraction of the exact brute-force top-10 neighbors retrieved by HNSW, averaged over the benchmark queries.

In [11]:
# Cell 1 - Imports
import ast
import heapq
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

print('05_benchmark.ipynb')
print('====================')
print('Benchmark environment ready.')

05_benchmark.ipynb
Benchmark environment ready.


In [12]:
# Cell 2 - Configuration
BASE_DIR = Path('..')
EMBEDDING_PATH = BASE_DIR / 'data' / 'embeddings' / 'embeddings.npy'
IDS_PATH = BASE_DIR / 'data' / 'embeddings' / 'ids.npy'
HNSW_NOTEBOOK = BASE_DIR / 'notebooks' / '04_fix_hnsw.ipynb'
TOP_K = 10
M = 12
EF_CONSTRUCTION = 100
EF_SEARCH_VALUES = [10, 20, 50, 100, 200]
QUERY_COUNT = 500
RANDOM_SEED = 42
print('Top-K:', TOP_K)
print('M:', M)
print('ef_construction:', EF_CONSTRUCTION)
print('ef_search values:', EF_SEARCH_VALUES)
print('Queries:', QUERY_COUNT)

Top-K: 10
M: 12
ef_construction: 100
ef_search values: [10, 20, 50, 100, 200]
Queries: 500


In [13]:
# Cell 3 - Load the existing full embeddings and IDs
embeddings = np.load(EMBEDDING_PATH)
ids = np.load(IDS_PATH)
assert embeddings.ndim == 2 and embeddings.shape[1] == 384
assert len(embeddings) == len(ids)
print(f'Number of vectors: {len(embeddings):,}')
print(f'Dimension: {embeddings.shape[1]}')
print(f'Embedding dtype: {embeddings.dtype}')
print(f'Number of IDs: {len(ids):,}')

Number of vectors: 119,921
Dimension: 384
Embedding dtype: float32
Number of IDs: 119,921


In [14]:
# Cell 4 - Verify embeddings
norms = np.linalg.norm(embeddings, axis=1)
print('Embedding validation')
print('--------------------')
print('Minimum norm:', norms.min())
print('Maximum norm:', norms.max())
print('Average norm:', norms.mean())
print('NaN values:', np.isnan(embeddings).sum())
print('Infinite values:', np.isinf(embeddings).sum())
assert np.isfinite(embeddings).all()

Embedding validation


--------------------
Minimum norm: 0.9999999
Maximum norm: 1.0000001
Average norm: 1.0
NaN values: 0
Infinite values: 0


In [15]:
# Cell 5 - Exact brute-force cosine search
def exact_search(query, vectors, vector_ids, k=TOP_K):
    scores = vectors @ query
    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return [(int(vector_ids[i]), float(scores[i])) for i in top_indices]

print('Exact search is the ground truth.')

Exact search is the ground truth.


In [16]:
# Cell 6 - Test exact search
query = embeddings[0]
results = exact_search(query, embeddings, ids, TOP_K)
print('Exact Search')
print('============')
for rank, (doc_id, score) in enumerate(results, 1):
    print(f'Rank {rank:2d} | ID: {doc_id:6d} | Similarity: {score:.4f}')

Exact Search
Rank  1 | ID:      1 | Similarity: 1.0000
Rank  2 | ID:     10 | Similarity: 0.9671
Rank  3 | ID:   3807 | Similarity: 0.5746
Rank  4 | ID:   8755 | Similarity: 0.5269
Rank  5 | ID:  10600 | Similarity: 0.5211
Rank  6 | ID:  38064 | Similarity: 0.5159
Rank  7 | ID:  35526 | Similarity: 0.5092
Rank  8 | ID:  72722 | Similarity: 0.5057
Rank  9 | ID:  72749 | Similarity: 0.5008
Rank 10 | ID:  89692 | Similarity: 0.4994


In [19]:
# Cell 7 - Load the existing handwritten HNSW implementation
from dataclasses import dataclass, field

def notebook_code_cells(path):
    text = path.read_text(encoding='utf-8-sig')
    if text.lstrip().startswith('<VSCode.Cell'):
        return re.findall(r'<VSCode\.Cell[^>]*language="python"[^>]*>\n?(.*?)</VSCode\.Cell>', text, flags=re.DOTALL)
    notebook = json.loads(text)
    return [''.join(cell.get('source', [])) for cell in notebook.get('cells', []) if cell.get('cell_type') == 'code']

def load_existing_hnsw(path):
    namespace = {
        '__name__': 'existing_hnsw',
        'np': np,
        'heapq': heapq,
        'time': time,
        'dataclass': dataclass,
        'field': field,
        'RANDOM_SEED': RANDOM_SEED,
        'cosine_similarity': lambda a, b: float(np.dot(a, b)),
    }
    sources = notebook_code_cells(path)
    selected = []
    wanted = ('class HNSWNode:', 'class HNSWIndex:', 'HNSWIndex.search =', 'def validate_graph(self):', 'HNSWIndex.validate_graph =')
    for source in sources:
        if any(marker in source for marker in wanted):
            selected.append(source)
    for source in selected:
        tree = ast.parse(source)
        exec(compile(tree, str(path), 'exec'), namespace)
    if 'HNSWIndex' not in namespace:
        raise RuntimeError(f'HNSWIndex was not found in {path}')
    return namespace['HNSWIndex']

HNSWIndex = load_existing_hnsw(HNSW_NOTEBOOK)
print(f'Loaded handwritten HNSWIndex from {HNSW_NOTEBOOK}')
print('No external vector index library is used.')

HNSW methods ready.
Loaded handwritten HNSWIndex from ..\notebooks\04_fix_hnsw.ipynb
No external vector index library is used.


In [20]:
# Cell 8 - Build HNSW once
print('Building HNSW once on the full dataset...')
print('This is the expensive cell.')
start = time.perf_counter()
hnsw = HNSWIndex(M=M, ef_construction=EF_CONSTRUCTION, ef_search=EF_SEARCH_VALUES[-1], seed=RANDOM_SEED)
for i, vector in enumerate(embeddings):
    hnsw.insert(vector, int(ids[i]))
    if (i + 1) % 10_000 == 0:
        elapsed = time.perf_counter() - start
        print(f'Inserted {i + 1:,}/{len(embeddings):,} in {elapsed:.1f}s')
build_time = time.perf_counter() - start
print(f'Vectors inserted: {len(embeddings):,}')
print(f'Build time: {build_time:.3f} seconds')

Building HNSW once on the full dataset...
This is the expensive cell.
Inserted 10,000/119,921 in 25.3s
Inserted 20,000/119,921 in 61.6s
Inserted 30,000/119,921 in 94.8s
Inserted 40,000/119,921 in 125.3s
Inserted 50,000/119,921 in 156.4s
Inserted 60,000/119,921 in 187.5s
Inserted 70,000/119,921 in 219.8s
Inserted 80,000/119,921 in 251.7s
Inserted 90,000/119,921 in 283.9s
Inserted 100,000/119,921 in 317.8s
Inserted 110,000/119,921 in 351.1s
Vectors inserted: 119,921
Build time: 383.742 seconds


In [21]:
# Cell 9 - Basic HNSW information and graph validation
print('HNSW Structure')
print('==============')
print('Entry point:', hnsw.entry_point)
print('Maximum level:', hnsw.max_level)
if hasattr(hnsw, 'graph'):
    print('Number of layers:', len(hnsw.graph))
    for level, layer in enumerate(hnsw.graph):
        print(f'Layer {level}: {len(layer):,} nodes')

if hasattr(hnsw, 'validate_graph'):
    validation = hnsw.validate_graph()
    print('Graph valid:', validation.get('valid'))
    if validation.get('issues'):
        print('Validation issues:')
        for issue in validation['issues'][:10]:
            print(' -', issue)
else:
    validation = {'valid': None, 'issues': []}
    print('validate_graph() is not available.')

HNSW Structure
Entry point: 9327
Maximum level: 16
Graph valid: False
Validation issues:
 - ('unreachable nodes', 0, [722, 735, 1006, 1033, 1142, 1154, 1159, 1160, 1205, 1219])
 - ('unreachable nodes', 1, [171, 576, 1006, 1033, 1142, 1154, 1269, 1324, 1469, 1537])
 - ('unreachable nodes', 2, [1698, 1923, 2573, 2602, 5606, 6282, 7463, 8485, 8640, 11392])
 - ('unreachable nodes', 3, [1923, 2152, 6469, 8212, 8251, 11725, 14648, 19510, 20052, 25453])
 - ('unreachable nodes', 4, [1923, 4572, 6535, 7441, 13876, 28770, 41981, 55763, 57910, 60863])
 - ('unreachable nodes', 5, [13876, 35624, 54922, 57910])
 - ('unreachable nodes', 6, [19482])


In [22]:
# Cell 10 - Reproducible benchmark queries
rng = np.random.default_rng(RANDOM_SEED)
query_indices = rng.choice(len(embeddings), size=min(QUERY_COUNT, len(embeddings)), replace=False)
queries = embeddings[query_indices]
print(f'Benchmark queries: {len(queries)}')
print('Query seed:', RANDOM_SEED)

Benchmark queries: 500
Query seed: 42


In [23]:
# Cell 11 - Compute exact ground truth
print('Computing exact ground truth...')
ground_truth = []
start = time.perf_counter()
for query in queries:
    ground_truth.append([doc_id for doc_id, _ in exact_search(query, embeddings, ids, TOP_K)])
ground_truth_time = time.perf_counter() - start
print(f'Ground truth generated in {ground_truth_time:.3f} seconds')

Computing exact ground truth...
Ground truth generated in 3.622 seconds


In [24]:
# Cell 12 - Normalize the existing HNSW search return format
def result_ids(search_results, index):
    found = []
    for item in search_results:
        if isinstance(item, dict):
            if 'id' in item:
                found.append(int(item['id']))
            elif 'index' in item:
                found.append(int(index.ids[item['index']]))
            continue
        if isinstance(item, (tuple, list)) and len(item) == 2:
            first, second = item
            if isinstance(second, (int, np.integer)) and hasattr(index, 'ids'):
                found.append(int(index.ids[int(second)]))
            elif isinstance(first, (int, np.integer)):
                found.append(int(first))
    return found

def hnsw_search_ids(index, query, k, ef_search):
    if hasattr(index, 'ef_search'):
        index.ef_search = int(ef_search)
    try:
        raw = index.search(query, k=k, ef_search=ef_search)
    except TypeError:
        try:
            raw = index.search(query, k=k)
        except TypeError:
            raw = index.search(query, k)
    return result_ids(raw, index)[:k]

def calculate_recall(predicted, actual, k=TOP_K):
    return len(set(predicted[:k]) & set(actual[:k])) / k

print('Search adapter ready for the existing implementation.')

Search adapter ready for the existing implementation.


In [25]:
# Cell 13 - Benchmark exact search latency
exact_times = []
for query in queries:
    start = time.perf_counter()
    exact_search(query, embeddings, ids, TOP_K)
    exact_times.append((time.perf_counter() - start) * 1000)
exact_avg = float(np.mean(exact_times))
exact_median = float(np.median(exact_times))
print(f'Exact average: {exact_avg:.3f} ms')
print(f'Exact median:  {exact_median:.3f} ms')

Exact average: 7.000 ms
Exact median:  6.908 ms


In [26]:
# Cell 14 - Benchmark HNSW ef_search trade-off
benchmark_results = []
print('HNSW Benchmark')
print('==============')
for ef in EF_SEARCH_VALUES:
    start = time.perf_counter()
    predicted = [hnsw_search_ids(hnsw, query, TOP_K, ef) for query in queries]
    elapsed = time.perf_counter() - start
    latencies = elapsed / len(queries) * 1000
    recalls = [calculate_recall(p, truth, TOP_K) for p, truth in zip(predicted, ground_truth)]
    recall = float(np.mean(recalls))
    benchmark_results.append({
        'dataset_size': len(embeddings),
        'M': M,
        'ef_construction': EF_CONSTRUCTION,
        'ef_search': ef,
        'build_time_sec': build_time,
        'exact_avg_latency_ms': exact_avg,
        'hnsw_avg_latency_ms': latencies,
        'recall_at_10': recall,
        'speedup': exact_avg / latencies if latencies else np.nan,
    })
    print(f'ef_search={ef:3d} | Recall@10={recall:.2%} | HNSW={latencies:.3f} ms | Speedup={exact_avg / latencies:.2f}x')

HNSW Benchmark
ef_search= 10 | Recall@10=67.48% | HNSW=0.642 ms | Speedup=10.90x
ef_search= 20 | Recall@10=73.68% | HNSW=0.750 ms | Speedup=9.34x
ef_search= 50 | Recall@10=80.90% | HNSW=1.144 ms | Speedup=6.12x
ef_search=100 | Recall@10=85.10% | HNSW=1.725 ms | Speedup=4.06x
ef_search=200 | Recall@10=88.98% | HNSW=2.841 ms | Speedup=2.46x


In [27]:
# Cell 15 - Results table
benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df.style.format({
    'build_time_sec': '{:.3f}',
    'exact_avg_latency_ms': '{:.3f}',
    'hnsw_avg_latency_ms': '{:.3f}',
    'recall_at_10': '{:.2%}',
    'speedup': '{:.2f}x',
}))

,dataset_size,M,ef_construction,ef_search,build_time_sec,exact_avg_latency_ms,hnsw_avg_latency_ms,recall_at_10,speedup
0,119921,12,100,10,383.742,7.000,0.642,67.48%,10.90x
1,119921,12,100,20,383.742,7.000,0.750,73.68%,9.34x
2,119921,12,100,50,383.742,7.000,1.144,80.90%,6.12x
3,119921,12,100,100,383.742,7.000,1.725,85.10%,4.06x
4,119921,12,100,200,383.742,7.000,2.841,88.98%,2.46x


In [28]:
# Cell 16 - Best tested configuration
best_row = benchmark_df.loc[benchmark_df['recall_at_10'].idxmax()]
print('BEST CONFIGURATION')
print('==================')
print(f'M: {int(best_row["M"])}')
print(f'ef_construction: {int(best_row["ef_construction"])}')
print(f'ef_search: {int(best_row["ef_search"])}')
print(f'Recall@10: {best_row["recall_at_10"]:.2%}')
print(f'HNSW latency: {best_row["hnsw_avg_latency_ms"]:.3f} ms')
print(f'Speedup: {best_row["speedup"]:.2f}x')

BEST CONFIGURATION
M: 12
ef_construction: 100
ef_search: 200
Recall@10: 88.98%
HNSW latency: 2.841 ms
Speedup: 2.46x


In [29]:
# Cell 17 - Demonstrate a real vector query
query_index = 0
query = queries[query_index]
results = hnsw_search_ids(hnsw, query, 5, int(best_row['ef_search']))
print('Example Vector Search')
print('=====================')
for rank, doc_id in enumerate(results, 1):
    score = float(embeddings[np.where(ids == doc_id)[0][0]] @ query)
    print(f'Rank {rank} | ID: {doc_id} | Score: {score:.4f}')

Example Vector Search
Rank 1 | ID: 15162 | Score: 0.6009
Rank 2 | ID: 15441 | Score: 0.5611
Rank 3 | ID: 55592 | Score: 0.5471
Rank 4 | ID: 17983 | Score: 0.5069
Rank 5 | ID: 92603 | Score: 0.4935


In [30]:
# Cell 18 - Save benchmark results
RESULTS_PATH = BASE_DIR / 'data' / 'benchmark_results.csv'
benchmark_df.to_csv(RESULTS_PATH, index=False)
print('Benchmark results saved to:')
print(RESULTS_PATH)

Benchmark results saved to:
..\data\benchmark_results.csv


In [31]:
# Cell 19 - Final results
print()
print('=' * 60)
print('FINAL VECTOR DATABASE BENCHMARK')
print('=' * 60)
print(f'Dataset size       : {len(embeddings):,} vectors')
print(f'Vector dimension    : {embeddings.shape[1]}')
print(f'M                  : {int(best_row["M"])}')
print(f'ef_construction    : {int(best_row["ef_construction"])}')
print(f'ef_search          : {int(best_row["ef_search"])}')
print(f'Build time         : {best_row["build_time_sec"]:.3f} sec')
print(f'Exact latency      : {best_row["exact_avg_latency_ms"]:.3f} ms')
print(f'HNSW latency       : {best_row["hnsw_avg_latency_ms"]:.3f} ms')
print(f'Speedup            : {best_row["speedup"]:.2f}x')
print(f'Recall@10          : {best_row["recall_at_10"]:.2%}')
print(f'Graph validation   : {validation.get("valid")}')
print('=' * 60)


FINAL VECTOR DATABASE BENCHMARK
Dataset size       : 119,921 vectors
Vector dimension    : 384
M                  : 12
ef_construction    : 100
ef_search          : 200
Build time         : 383.742 sec
Exact latency      : 7.000 ms
HNSW latency       : 2.841 ms
Speedup            : 2.46x
Recall@10          : 88.98%
Graph validation   : False


In [32]:
# Cell 20 - Plain-English interpretation
recall_pct = best_row['recall_at_10'] * 100
print('PROJECT RESULT IN SIMPLE WORDS')
print('===============================')
print(f'We indexed {len(embeddings):,} document embeddings.')
print(f'Each embedding has {embeddings.shape[1]} dimensions.')
print('Exact search checks every vector and provides the ground-truth top-10.')
print('HNSW searches a graph and avoids checking every vector for each query.')
print(f'Recall@10 was {recall_pct:.1f}%, meaning HNSW retrieved about {recall_pct / 10:.2f} of the 10 exact neighbors on average.')
print(f'HNSW was approximately {best_row["speedup"]:.2f}x faster for this benchmark.')
print('Higher ef_search generally increases recall and latency together.')

PROJECT RESULT IN SIMPLE WORDS
We indexed 119,921 document embeddings.
Each embedding has 384 dimensions.
Exact search checks every vector and provides the ground-truth top-10.
HNSW searches a graph and avoids checking every vector for each query.
Recall@10 was 89.0%, meaning HNSW retrieved about 8.90 of the 10 exact neighbors on average.
HNSW was approximately 2.46x faster for this benchmark.
Higher ef_search generally increases recall and latency together.


In [33]:
# Cell 21 - Generate complete project reports
# This cell reads only notebooks 01-05 from the active notebooks folder.
# It excludes notebooks/not_needed and does not execute or modify any notebook.

import json
from datetime import datetime

ACTIVE_NOTEBOOKS = [
    BASE_DIR / 'notebooks' / '01_data_exploration.ipynb',
    BASE_DIR / 'notebooks' / '02_embedding_collab.ipynb',
    BASE_DIR / 'notebooks' / '03_exact_search.ipynb',
    BASE_DIR / 'notebooks' / '04_fix_hnsw.ipynb',
    BASE_DIR / 'notebooks' / '05_benchmark.ipynb',
]

BENCHMARK_REPORT_PATH = BASE_DIR / 'benchmark.txt'
EXPLANATION_REPORT_PATH = BASE_DIR / 'explanation.txt'


def load_notebook(path):
    with open(path, 'r', encoding='utf-8-sig') as file:
        return json.load(file)


def source_text(cell):
    source = cell.get('source', '')
    return ''.join(source) if isinstance(source, list) else str(source)


def output_text(output):
    output_type = output.get('output_type', '')
    if output_type == 'stream':
        value = output.get('text', '')
    elif output_type in {'execute_result', 'display_data', 'error'}:
        data = output.get('data', {})
        value = data.get('text/plain', '')
        if not value and output_type == 'error':
            value = '\\n'.join(output.get('traceback', []))
    else:
        value = output.get('text', '')

    if isinstance(value, list):
        return ''.join(value)
    return str(value)


def cell_heading(cell, cell_number):
    source = source_text(cell).strip()
    first_line = source.splitlines()[0].strip() if source else ''
    if first_line.startswith('#'):
        return first_line.lstrip('#').strip()
    if cell.get('cell_type') == 'markdown':
        for line in source.splitlines():
            if line.strip():
                return line.lstrip('#').strip()
    return f'{cell.get("cell_type", "unknown").title()} cell'


def explain_cell(cell, cell_number):
    source = source_text(cell).strip()
    if not source:
        return 'This cell is empty.'
    if cell.get('cell_type') == 'markdown':
        return 'This markdown cell documents the project workflow or the purpose of the nearby code.'

    lines = source.splitlines()
    definitions = []
    imports = []
    actions = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith(('def ', 'class ')):
            definitions.append(stripped.split('(')[0].rstrip(':'))
        if stripped.startswith(('import ', 'from ')):
            imports.append(stripped)
        if any(token in stripped for token in ('np.load', 'read_parquet', 'read_json', 'to_csv', 'encode(', 'insert(', 'search(')):
            actions.append(stripped)

    parts = []
    if imports:
        parts.append('It imports ' + ', '.join(imports[:4]) + '.')
    if definitions:
        parts.append('It defines ' + ', '.join(definitions[:8]) + '.')
    if actions:
        parts.append('Important operations include: ' + '; '.join(actions[:5]) + '.')
    if not parts:
        parts.append('It performs the calculations, configuration, display, or checks shown in the source code.')
    return ' '.join(parts)


benchmark_sections = []
explanation_sections = [
    'VECTOR DATABASE PROJECT EXPLANATION',
    '====================================',
    'Generated: ' + datetime.now().isoformat(timespec='seconds'),
    '',
    'This report explains every cell in the active notebooks 01 through 05.',
    'The notebooks under notebooks/not_needed are intentionally excluded.',
    '',
]

for notebook_path in ACTIVE_NOTEBOOKS:
    if not notebook_path.exists():
        benchmark_sections.append(f'MISSING NOTEBOOK: {notebook_path.name}')
        explanation_sections.append(f'WARNING: {notebook_path.name} was not found.')
        continue

    notebook = load_notebook(notebook_path)
    notebook_name = notebook_path.name
    cells = notebook.get('cells', [])

    benchmark_sections.extend([
        '',
        '=' * 70,
        notebook_name.upper(),
        '=' * 70,
    ])
    explanation_sections.extend([
        '',
        '=' * 70,
        notebook_name.upper(),
        '=' * 70,
    ])

    for cell_number, cell in enumerate(cells, 1):
        heading = cell_heading(cell, cell_number)
        source = source_text(cell).strip()
        explanation = explain_cell(cell, cell_number)

        explanation_sections.extend([
            '',
            f'CELL {cell_number}: {heading}',
            '-' * 70,
            explanation,
            '',
            'Cell source:',
            source,
        ])

        outputs = [output_text(output).strip() for output in cell.get('outputs', [])]
        outputs = [output for output in outputs if output]
        if outputs:
            benchmark_sections.extend([
                '',
                f'CELL {cell_number}: {heading}',
                '-' * 70,
            ])
            benchmark_sections.extend(outputs)

    if not any(cell.get('outputs') for cell in cells):
        benchmark_sections.append('No saved output was found in this notebook.')

benchmark_sections.extend([
    '',
    '=' * 70,
    'CURRENT CELL 21 STATUS',
    '=' * 70,
    'benchmark.txt contains saved human-readable outputs from active notebooks 01-05.',
    'explanation.txt contains a cell-by-cell explanation and source listing for active notebooks 01-05.',
])

BENCHMARK_REPORT_PATH.write_text('\\n'.join(benchmark_sections) + '\\n', encoding='utf-8')
EXPLANATION_REPORT_PATH.write_text('\\n'.join(explanation_sections) + '\\n', encoding='utf-8')

print('Reports generated successfully.')
print('Benchmark report:', BENCHMARK_REPORT_PATH)
print('Explanation report:', EXPLANATION_REPORT_PATH)
print('Included notebooks:', ', '.join(path.name for path in ACTIVE_NOTEBOOKS if path.exists()))

Reports generated successfully.
Benchmark report: ..\benchmark.txt
Explanation report: ..\explanation.txt
Included notebooks: 01_data_exploration.ipynb, 02_embedding_collab.ipynb, 03_exact_search.ipynb, 04_fix_hnsw.ipynb, 05_benchmark.ipynb
